# Tutorial 15 — Polarization-Like Orientation Maps

Synthetic forward and inverse examples with known ground truth.


In [ ]:
LANGUAGE = "en"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "biomechanics_tutorials").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))
import biomechanics_tutorials

print(Path(biomechanics_tutorials.__file__).resolve())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from biomechanics_tutorials.polarization_orientation import (
    PolarizationParameters,
    simulate_polarization_series,
    recover_orientation_harmonic,
    orientation_error_deg,
)

In [ ]:
shape = (96, 96)
yy, xx = np.indices(shape)
orientation = 0.7 * np.arctan2(yy - 48, xx - 48)
retardance = 0.5 + 1.2 * np.exp(-((xx - 48) ** 2 + (yy - 48) ** 2) / 1200)
mask = ((xx - 48) ** 2 + (yy - 48) ** 2) < 42**2
amplitude = mask.astype(float)
angles = np.linspace(0, np.pi, 12, endpoint=False)
series = simulate_polarization_series(
    orientation, retardance, angles, amplitude, PolarizationParameters(read_noise_std=0.01), seed=15
)
recovery = recover_orientation_harmonic(series.images, angles)
error = orientation_error_deg(recovery.orientation_rad, orientation)
print(float(np.median(error[mask])))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, array, title, cmap in zip(
    axes,
    [series.images[0], np.rad2deg(orientation), np.rad2deg(recovery.orientation_rad), error],
    ["frame", "truth", "recovered", "error"],
    ["gray", "twilight", "twilight", "magma"],
):
    im = ax.imshow(np.where(mask, array, np.nan), cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.65)
plt.tight_layout()
plt.show()